# Soccer: Sports Analytics

**Sports Analytics Club at UT Dallas · Workshop 2**

**Edit this cell** marks your tasks. **Run this cell** means the code is ready; leave it as provided.
Work from top to bottom and replace `____` using the example.

> **Sport in 30 seconds**
> - One row = one shot at goal.
> - `goal` records whether the shot scored.
> - `distance_pct` measures distance from goal in normalized pitch coordinates.
> - `head_or_body` records Wyscout’s head/body contact tag.

## Setup

Connect to a Colab **CPU** runtime and run both collapsed cells. The dataset is included and its checksum is checked.

In [ ]:
#@title Import libraries — run this cell
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, ConfusionMatrixDisplay

SEED = 42
plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)


In [ ]:
#@title Prepared dataset — run this cell (expand only if curious)
# Prepared data snapshot. No network calls. Instructor rebuilds with scripts/prepare_data.py.
import base64, hashlib, io, zipfile
EXPECTED_SHA256 = {'soccer_data.csv': '796e1bf5f02ac0337b9018d046ab82ba3614bd5f57cd3e4838862af21f94f7ae'}
SNAPSHOT_B64 = "..."
candidates = [Path('data'), Path('../data')]
DATA_DIR = next((p for p in candidates if all((p/n).is_file() for n in EXPECTED_SHA256)), None)
if DATA_DIR is None:
    DATA_DIR = Path('workshop2_data')
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(SNAPSHOT_B64))) as archive:
        for name in EXPECTED_SHA256:
            (DATA_DIR/name).write_bytes(archive.read(name))
for name, expected in EXPECTED_SHA256.items():
    actual = hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()
    assert actual == expected, f'{name} differs from the workshop snapshot. Restore the supplied CSV.'
print('Ready: 1 verified CSV.')


## First look

Run this cell to preview the data.

In [ ]:
shots = pd.read_csv(DATA_DIR/'soccer_data.csv')
for name, frame in [('Soccer', shots)]:
    print(name, 'rows/columns:', frame.shape, '| missing cells:', frame.isna().sum().sum())
    display(frame.head(3))


| Column | Plain English |
|---|---|
| `goal` | 1 = scored; 0 = did not score — our target |
| `distance_pct` | Distance from goal in normalized coordinates, not metres |
| `head_or_body` | 1 = Wyscout head/body tag; 0 = that tag is absent (not just a header indicator) |

## Estimate goal probability

We split by match so shots from the same game do not appear in both training and testing.

In [ ]:
#@title Run this cell — prepare the match split and scaling
# Distance uses normalized coordinates, not metres; head_or_body is not just headers.
class_features = ['distance_pct', 'head_or_body']
# Keep all shots from a match together. This tests other matches, not future seasons.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(shots, groups=shots['match_id']))
shot_train, shot_test = shots.iloc[train_idx].copy(), shots.iloc[test_idx].copy()
assert set(shot_train.match_id).isdisjoint(set(shot_test.match_id))
print('Train matches:', shot_train.match_id.nunique(), '| Test matches:', shot_test.match_id.nunique())
print('Train shots:', len(shot_train), '| Test shots:', len(shot_test))
print(f'Training goal rate: {shot_train.goal.mean():.1%}')

# Learn scaling from TRAIN only. Reuse it on TEST.
shot_scaler = StandardScaler()
shot_X_train = shot_scaler.fit_transform(shot_train[class_features])
shot_X_test = shot_scaler.transform(shot_test[class_features])


### Worked example — run this cell

In [ ]:
# Worked example: learn from training shots, then estimate three goal probabilities.
example_classifier = LogisticRegression(max_iter=1000, random_state=SEED)
example_classifier.fit(shot_X_train, shot_train['goal'])
print(example_classifier.predict_proba(shot_X_test[:3])[:, 1])

### S1 · Edit this cell

Follow the example: use `LogisticRegression`, `shot_X_train`, and `shot_train['goal']`.

In [ ]:
shot_model = ____(max_iter=1000, random_state=SEED)
shot_model.fit(____, ____)


### S2 · Edit this cell

**Precision:** of the shots predicted as goals, how many actually scored?

**Recall:** of all actual goals, how many did the model catch?

**Predict first:** If we lower the threshold, do you expect us to catch more goals or fewer goals?

Run at 0.50, then change to 0.15. Each time, run your edit and the next output cell.

In [ ]:
# Try it yourself
threshold = 0.50   # Change to 0.15


In [ ]:
#@title Run this cell — show scores and the confusion matrix
goal_probabilities = shot_model.predict_proba(shot_X_test)[:, 1]
assert 0 < threshold < 1
shot_predictions = (goal_probabilities >= threshold).astype(int)
always_no_goal = np.zeros(len(shot_test), dtype=int)
# Accuracy counts all correct decisions; precision is shown as 0 if no goals are flagged.
def classification_scores(name, predictions):
    return {'model': name, 'accuracy': accuracy_score(shot_test.goal, predictions),
            'goal_precision': precision_score(shot_test.goal, predictions, zero_division=0),
            'goal_recall': recall_score(shot_test.goal, predictions, zero_division=0)}
class_results = pd.DataFrame([
    classification_scores('Always no goal', always_no_goal),
    classification_scores(f'Logistic, threshold={threshold:.2f}', shot_predictions)])
display(class_results.round(3))
ConfusionMatrixDisplay.from_predictions(shot_test.goal, shot_predictions,
    display_labels=['No goal', 'Goal'], cmap='Blues', colorbar=False)
plt.title(f'Soccer test shots: threshold {threshold:.2f}')
plt.tight_layout(); plt.show()


**At 0.15, do we miss fewer goals but raise more false alarms?**

Matrix rows are actual outcomes; columns are predictions. Missed goals are actual goals predicted as no goal.

## Optional extension: distance and probability

This curve holds `head_or_body = 0` to compare distances for shots **without** the head/body tag. That keeps contact type fixed. The threshold changes decisions, not these probabilities.

In [ ]:
#@title Run this cell — draw the probability curve
# The threshold changes decisions, not this probability curve.
# These are model scenarios; missing defenders, angle, and game context limit the model.
distance_grid = np.linspace(1, 45, 100)
scenario = pd.DataFrame({'distance_pct': distance_grid, 'head_or_body': 0})
curve = shot_model.predict_proba(shot_scaler.transform(scenario[class_features]))[:, 1]
plt.plot(distance_grid, curve, color='#157F78', linewidth=3)
plt.xlabel('Distance proxy (normalized units, not metres)')
plt.ylabel('Model goal probability')
plt.title('Model response for shots without the head/body tag')
plt.ylim(0, 1); plt.tight_layout(); plt.show()


<details>
<summary>Sources and definitions</summary>

Prepared files and complete column definitions: `data/DATA_SOURCES.md` and `data/DATA_DICTIONARY.md` in the package.
Raw-source URLs, checksums, and row-filter audit: `data/manifest.json`.

- **Soccer:** Wyscout data, Pappalardo & Massucco (2019), [Events](https://doi.org/10.6084/m9.figshare.7770599), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Derived shot subset and features prepared for this workshop. [Dataset paper](https://doi.org/10.1038/s41597-019-0247-7), [tag definitions](https://doi.org/10.6084/m9.figshare.11743818).

- **Methods:** [Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) · [scaling](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) · [precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html) · [recall](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)

</details>